# AMEVA-STT-Trainer Google Colab 통합 실행 환경

이 노트북은 **구글 코랩 무료 GPU(T4)**를 활용하여 AMEVA-STT 모델의 데이터셋 구축부터 학습까지 모바일 데이터 소모 없이 수행하기 위한 통합 실행 파일입니다.

### 🌟 주요 이점
1. **0MB 데이터 소모**: 코랩 서버가 유튜브에서 데이터를 직접 수집하므로 모바일 핫스팟/요금제 데이터가 소비되지 않습니다.
2. **GPU 가속**: CPU 대비 약 100배 빠른 학습 속도 (20~30분 소요).
3. **자동 동기화**: 최종 결과물(수십 MB의 LoRA 어댑터)은 구글 드라이브를 통해 내 PC로 자동 동기화됩니다.

## 1. 구글 드라이브 마운트 및 경로 이동

In [ ]:
from google.colab import drive
import os

# 구글 드라이브 연결
drive.mount('/content/drive')

# 프로젝트 폴더 경로 지정
# 구글 드라이브의 "내 드라이브/AMEVA-STT-Trainer" 경로에 코드를 업로드해 두어야 합니다.
PROJECT_PATH = "/content/drive/MyDrive/AMEVA-STT-Trainer"

if not os.path.exists(PROJECT_PATH):
    print(f"⚠️ [오류] {PROJECT_PATH} 경로를 찾을 수 없습니다.")
    print("구글 드라이브에 프로젝트 폴더를 업로드했는지 확인해 주세요.")
else:
    %cd "$PROJECT_PATH"
    print(f"✅ 프로젝트 디렉터리로 이동 완료: {os.getcwd()}")

## 2. 필수 라이브러리 및 환경 구축
코랩에 이미 기본으로 설치되어 있는 PyTorch 외에 학습에 필요한 핵심 패키지들을 설치합니다. (Windows 전용 패키지 및 UI 패키지는 제외하여 신속하게 빌드합니다.)

In [ ]:
# 필수 패키지 고속 설치
!pip install transformers datasets peft accelerate evaluate jiwer librosa soundfile pydub webvtt-py yt-dlp pandas pyyaml rich psutil python-docx matplotlib

# 오디오 처리를 위한 ffmpeg 시스템 패키지 설치
!apt-get update && apt-get install -y ffmpeg

## 3. [추천] 슈카월드 300개 비디오 통합 원클릭 실행
아래 코드를 실행하면 **슈카월드 300개 영상 수집 -> 2,000스텝 학습 -> GGUF 양자화** 전 과정을 동기식(실시간 로그 출력)으로 한 번에 원스톱 실행합니다. 데이터 요금 소비는 0MB에 가깝습니다.

In [ ]:
# 슈카월드 300개 마스터 파이프라인 동기식 원스톱 실행
!python scripts/run_syuka_300_pipeline_sync.py

## 4. [개별 실행] [1단계] 데이터셋 구축 (유튜브 다운로드 및 전처리)
직접 개별 단계별로 세부 튜닝하여 실행하고 싶다면 이 단계들을 순차적으로 실행하세요.
여기에 다운로드하고 싶은 유튜브 주소를 입력하여 코랩 서버가 직접 데이터를 수집하게 만듭니다. (데이터 요금 소모 없음)

* **주의**: 만약 이미 로컬에서 빌드된 데이터셋이 구글 드라이브에 올라가 있다면 이 단계를 건너뛰고 바로 학습 단계로 가셔도 됩니다.

In [ ]:
# 다운로드할 유튜브 채널/영상 주소 설정
CHANNEL_URL = "https://www.youtube.com/@syukaworld/videos" # 예시: 슈카월드
MAX_VIDEOS = 5 # 테스트를 위해 영상 개수를 제한합니다. (필요시 수정)

# 1단계 데이터 구축 스크립트 호출 (코랩 서버가 직접 수집하므로 모바일 데이터 소모 0)
!python scripts/01_build_dataset.py --channel_url "$CHANNEL_URL" --max_videos $MAX_VIDEOS

## 5. [개별 실행] [2단계] Whisper LoRA GPU 학습 시작
구글 코랩의 고성능 GPU 가속을 활용해 학습을 시작합니다.

* **📌 중요**: 상단 메뉴의 **[런타임] -> [런타임 유형 변경]**에서 하드웨어 가속기가 **T4 GPU** 이상으로 지정되어 있는지 반드시 확인하세요!
* 이전 체크포인트가 존재한다면 코드가 이를 자동으로 인지하여 이어서 학습(Resume)하며, 학습률도 안전하게 기존의 50%로 자동 감쇄되고 Gradient Clipping 1.0 기능이 가동되어 안정적인 가속 학습을 진행합니다.

In [ ]:
# 2단계 학습 스크립트 실행 (--skip을 주어 1단계 중복 실행 방지)
# task-id가 있다면 지정해주고, 없다면 기본 설정을 사용합니다.
TASK_ID = "a28cf223-f1c6-4c5e-914a-8f828bfe721f" # 예시: 현재 태스크 ID

!python scripts/02_start_training.py --task-id "$TASK_ID" --skip

## 6. [개별 실행] [3단계] 최종 모델 병합 및 GGUF 변환 (선택 사항)
학습이 끝난 LoRA 가중치를 베이스 모델과 결합하여 배포용 단일 모델 또는 C++ 추론용 GGUF 모델로 내보냅니다.

In [ ]:
# 3단계 최종 모델 추출 실행
# !python scripts/03_export_model.py --task-id "$TASK_ID"